In [ ]:
# Imports
from pathlib import Path
import os
import glob
import re
import itertools

import numpy as np
import pandas as pd



In [ ]:
# Load rule datasets
all_rules_crm_path = Path("5_analysis/coverage=10/random/combined_sorted_all.csv")
# all_rules_dt_path = Path("5_analysis/dt/rules_dt.csv")
all_rules_ripperk_path = Path("5_analysis/ripperk/rules_ripperk.csv")

all_rules_crm = pd.read_csv(all_rules_crm_path)
# all_rules_dt = pd.read_csv(all_rules_dt_path)
all_rules_ripperk = pd.read_csv(all_rules_ripperk_path)


In [ ]:
# Distinct encodings per labeling
unique_counts = (
    all_rules_crm
    .groupby("Labeling")["Feature Encoding"]
    .nunique()
    .reset_index(name="unique_encodings")
)

display(unique_counts)

,Labeling,unique_encodings
0,BPI15A_decl2,1
1,BPI15A_mr_tr,1
2,BPI15A_payload_560925,9
3,sepsis_decl,8
4,sepsis_mr_tr,13
5,sepsis_payload2,4
6,traffic_decl3,15
7,traffic_mr_tr,15
8,traffic_payload_Pay36,15


In [ ]:
# Normalize labeling values and align column names across dataframes

def _normalize_labeling_column(df: pd.DataFrame) -> None:
    """
    Normalize the labeling/Labeling column to one of:
    {'declare', 'sequential', 'payload'} based on substrings.
    Operates in-place.
    """
    for col in ("labeling", "Labeling"):
        if col in df.columns:
            lower = df[col].astype(str).str.lower()
            df[col] = np.select(
                [
                    lower.str.contains("decl", na=False),
                    lower.str.contains("mr_tr", na=False),
                    lower.str.contains("payload", na=False),
                ],
                ["declare", "sequential", "payload"],
                default=df[col],
            )

def _canonicalize_and_subset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize common column variants and return only the key columns
    if present: ['Dataset', 'Labeling', 'Feature Encoding', 'Rule'].
    """
    rename_map = {}
    for c in df.columns:
        cl = c.lower()
        if cl == "dataset":
            rename_map[c] = "Dataset"
        elif cl == "labeling":
            rename_map[c] = "Labeling"
        elif cl in {"feature encoding", "encoding"}:
            rename_map[c] = "Feature Encoding"
        elif cl == "rule":
            rename_map[c] = "Rule"
    df2 = df.rename(columns=rename_map)

    keep = [c for c in ["Dataset", "Labeling", "Feature Encoding", "Rule"] if c in df2.columns]
    return df2[keep].copy() if keep else df2.copy()

# Apply normalization and column alignment to all three dataframes
for name in ("all_rules_crm", "all_rules_ripperk"):
    df = globals()[name]
    _normalize_labeling_column(df)
    df = _canonicalize_and_subset(df)
    globals()[name] = df

# Quick check
display(all_rules_crm)
# display(all_rules_dt)
display(all_rules_ripperk)


,Dataset,Labeling,Feature Encoding,Rule
0,BPI15A,declare,payload,['monitoringResource|first|literal_binned_(560...
1,BPI15A,declare,payload,"['length_binned_(44.0, 101.0]'] --> Label"
2,BPI15A,declare,payload,"['length_binned_(1.999, 44.0]'] --> !Label"
3,BPI15A,declare,payload,"['org:resource|first|literal_binned_(560912.0,..."
4,BPI15A,declare,payload,['monitoringResource|first|literal_binned_(560...
...,...,...,...,...
604,traffic,payload,seq_combined_data,"['amount|first|continuous_binned_(-0.001, 33.6..."
605,traffic,payload,seq_combined_data,"['mr[Create Fine-complete, Send Fine-complete,..."
606,traffic,payload,seq_combined_data,"['mr[Create Fine-complete, Send Fine-complete]..."
607,traffic,payload,seq_combined_data,['mr[Send for Credit Collection-complete]_0.0'...


,Dataset,Labeling,Feature Encoding,Rule
0,BPI15A,declare,baseline,[01_HOOFD_011 = 1.0] --> Label
1,BPI15A,declare,bs_data,[01_HOOFD_011 = 1.0] --> Label
2,BPI15A,declare,bs_dwd,"[alternate_precedence:(01_HOOFD_011,01_HOOFD_0..."
3,BPI15A,declare,dec_data,"[choice:('01_HOOFD_010', '01_HOOFD_011') = -1...."
4,BPI15A,declare,dec_dwd,"[choice:('01_HOOFD_010', '01_HOOFD_011') = -1...."
...,...,...,...,...
438,traffic,payload,seq_combined_data,[paymentAmount|first|continuous_binned_(-0.001...
439,traffic,payload,seq_combined_data,[paymentAmount|first|continuous_binned_(-0.001...
440,traffic,payload,seq_combined_data,[paymentAmount|first|continuous_binned_(-0.001...
441,traffic,payload,seq_combined_data,[paymentAmount|first|continuous_binned_(-0.001...


## Splitting CRM

In [ ]:
# 1) Parse LHS and RHS from all_rules_crm["Rule"]

def extract_lhs_exact(rule_str: str) -> str:
    """Return the substring before '-->' (keeps quotes/brackets as-is)."""
    m = re.search(r"^(.*?)(?=\s*-->)", str(rule_str))
    return m.group(1) if m else str(rule_str)

def parse_rhs_label(rule_str: str):
    """Map RHS to 1 for 'Label', 0 for '!Label'; None if not present."""
    m = re.search(r"-->\s*(Label|!Label)", str(rule_str))
    if not m:
        return None
    return 1 if m.group(1) == "Label" else 0

crm_df = all_rules_crm.copy()
crm_df["LHS_features"] = crm_df["Rule"].apply(extract_lhs_exact)
crm_df["RHS_label"]    = crm_df["Rule"].apply(parse_rhs_label)

# 2) Split LHS into up to 3 features

def _find_outer_brackets_span(text: str):
    """Indices of the outermost [...] in text; returns (start, end)."""
    s = str(text)
    start = s.find("[")
    if start < 0:
        return None, None

    depth = 0
    in_s = in_d = esc = False
    end = None
    for i, ch in enumerate(s[start:], start):
        if esc:
            esc = False
            continue
        if ch == "\\":
            esc = True
            continue

        if in_s:
            if ch == "'":
                in_s = False
            continue
        if in_d:
            if ch == '"':
                in_d = False
            continue

        if ch == "'":
            in_s = True
            continue
        if ch == '"':
            in_d = True
            continue

        if ch == "[":
            depth += 1
            continue
        if ch == "]":
            depth -= 1
            if depth == 0:
                end = i
                break
    return (start, end)

def _split_top_level_commas(content: str):
    """Split on commas that are not inside quotes."""
    parts, curr = [], ""
    in_s = in_d = esc = False
    for ch in content:
        if esc:
            curr += ch
            esc = False
            continue
        if ch == "\\":
            curr += ch
            esc = True
            continue

        if in_s:
            curr += ch
            if ch == "'":
                in_s = False
            continue
        if in_d:
            curr += ch
            if ch == '"':
                in_d = False
            continue

        if ch == "'":
            curr += ch
            in_s = True
            continue
        if ch == '"':
            curr += ch
            in_d = True
            continue

        if ch == ",":
            parts.append(curr.strip())
            curr = ""
        else:
            curr += ch
    parts.append(curr.strip())
    return parts

def _strip_one_layer_quotes(s: str):
    """Remove a single pair of outer quotes if present."""
    s = s.strip()
    if len(s) >= 2 and ((s[0] == s[-1] == "'") or (s[0] == s[-1] == '"')):
        return s[1:-1]
    return s

def split_lhs_items(lhs_text: str):
    """
    Input like "['A', 'B', 'C']" or "['A']" → list ['A','B','C'].
    """
    s = str(lhs_text)
    start, end = _find_outer_brackets_span(s)
    if start is None or end is None:
        return []
    inner = s[start + 1 : end]  # inside [...]
    raw_items = _split_top_level_commas(inner)
    return [_strip_one_layer_quotes(x).strip() for x in raw_items if x != ""]

def _pad3(items):
    """Keep at most 3 items; right-pad with empty strings."""
    items = items[:3]
    return items + [""] * (3 - len(items))

lhs_split = crm_df["LHS_features"].apply(split_lhs_items).apply(_pad3)
lhs_df = pd.DataFrame(lhs_split.tolist(), columns=["feature_1_lhs", "feature_2_lhs", "feature_3_lhs"])

# 3) Assemble expanded table
cols_present = [c for c in ["Dataset", "Labeling", "Feature Encoding", "Rule", "LHS_features", "RHS_label"] if c in crm_df.columns]

all_rules_crm_expanded = pd.concat(
    [crm_df[cols_present].reset_index(drop=True), lhs_df.reset_index(drop=True)],
    axis=1,
).reset_index(drop=True)

# Sort if keys are available
sort_keys = [c for c in ["Dataset", "Labeling", "Feature Encoding"] if c in all_rules_crm_expanded.columns]
if sort_keys:
    all_rules_crm_expanded = (
        all_rules_crm_expanded.sort_values(by=sort_keys, ascending=True).reset_index(drop=True)
    )

all_rules_crm_expanded

,Dataset,Labeling,Feature Encoding,Rule,LHS_features,RHS_label,feature_1_lhs,feature_2_lhs,feature_3_lhs
0,BPI15A,declare,payload,['monitoringResource|first|literal_binned_(560...,['monitoringResource|first|literal_binned_(560...,1,monitoringResource|first|literal_binned_(56092...,,
1,BPI15A,declare,payload,"['length_binned_(44.0, 101.0]'] --> Label","['length_binned_(44.0, 101.0]']",1,"length_binned_(44.0, 101.0]",,
2,BPI15A,declare,payload,"['length_binned_(1.999, 44.0]'] --> !Label","['length_binned_(1.999, 44.0]']",0,"length_binned_(1.999, 44.0]",,
3,BPI15A,declare,payload,"['org:resource|first|literal_binned_(560912.0,...","['org:resource|first|literal_binned_(560912.0,...",0,"org:resource|first|literal_binned_(560912.0, 1...",,
4,BPI15A,declare,payload,['monitoringResource|first|literal_binned_(560...,['monitoringResource|first|literal_binned_(560...,0,monitoringResource|first|literal_binned_(56046...,,
...,...,...,...,...,...,...,...,...,...
604,traffic,sequential,seq_combined_data,"['mr[Create Fine-complete, Send Fine-complete,...","['mr[Create Fine-complete, Send Fine-complete,...",0,"mr[Create Fine-complete, Send Fine-complete, I...",,
605,traffic,sequential,seq_combined_data,['mr[Add penalty-complete]_1.0'] --> Label,['mr[Add penalty-complete]_1.0'],1,mr[Add penalty-complete]_1.0,,
606,traffic,sequential,seq_combined_data,['paymentAmount|first|continuous_binned_(-0.00...,['paymentAmount|first|continuous_binned_(-0.00...,0,"paymentAmount|first|continuous_binned_(-0.001,...",,
607,traffic,sequential,seq_combined_data,"['paymentAmount|first|continuous_binned_(35.0,...","['paymentAmount|first|continuous_binned_(35.0,...",1,"paymentAmount|first|continuous_binned_(35.0, 1...",,


In [ ]:
# ---------- Expand DT and RIPPERk rules: support up to 15 LHS features ----------

def extract_lhs_exact(rule_str: str) -> str:
    """Everything before the arrow '-->' (preserve characters exactly)."""
    m = re.search(r"^(.*?)(?=\s*-->)", str(rule_str))
    return m.group(1).strip() if m else str(rule_str).strip()

def parse_rhs_label(rule_str: str):
    """Return 1 for 'Label', 0 for '!Label', or None if not found."""
    m = re.search(r"-->\s*(Label|!Label)", str(rule_str))
    if not m:
        return None
    return 1 if m.group(1) == "Label" else 0

# Splitter for DT/RIPPERK: use logical-and '∧' (U+2227); also accept ASCII '&' as fallback.
_AND_SPLIT_RE = re.compile(r"\s*(?:∧|&)\s*")

def split_lhs_items_dt(lhs_text: str):
    """
    For DT/RIPPERK rule format, LHS looks like:
      [feature1 ∧ feature2 ∧ feature3 ∧ ...]
    We split on '∧' (and '&' as fallback), strip outer [ ], then trim items.
    """
    s = str(lhs_text).strip()
    if len(s) >= 2 and s[0] == '[' and s[-1] == ']':
        s = s[1:-1]
    if not s:
        return []
    parts = _AND_SPLIT_RE.split(s)
    return [p.strip() for p in parts if p.strip() != ""]

def _padN(items, n=15):
    items = items[:n]
    return items + [""] * (n - len(items))

def _expand_df_with_lhs_rhs(df_in: pd.DataFrame, name_hint: str, max_features: int = 15):
    """
    Given a dataframe with at least ['Rule'] column, produce an expanded version with:
      - LHS_features: exact text before -->
      - RHS_label: {1,0,None}
      - feature_1_lhs ... feature_{max_features}_lhs (split on ∧ / & for DT/RIPPERK)
    Keeps any of ['Dataset','Labeling','Feature Encoding','Rule'] that exist.
    """
    if "Rule" not in df_in.columns:
        raise KeyError(f"{name_hint}: expected a 'Rule' column.")

    df = df_in.copy()
    df["LHS_features"] = df["Rule"].apply(extract_lhs_exact)
    df["RHS_label"]    = df["Rule"].apply(parse_rhs_label)

    lhs_split = df["LHS_features"].apply(split_lhs_items_dt).apply(lambda xs: _padN(xs, max_features))
    feat_cols = [f"feature_{i}_lhs" for i in range(1, max_features+1)]
    lhs_df = pd.DataFrame(lhs_split.tolist(), columns=feat_cols)

    keep = [c for c in ["Dataset","Labeling","Feature Encoding","Rule","LHS_features","RHS_label"] if c in df.columns or c in ["LHS_features","RHS_label"]]
    expanded = pd.concat([df[keep].reset_index(drop=True), lhs_df.reset_index(drop=True)], axis=1)

    sort_keys = [c for c in ["Dataset","Labeling","Feature Encoding"] if c in expanded.columns]
    if sort_keys:
        expanded = expanded.sort_values(by=sort_keys, ascending=True).reset_index(drop=True)
    return expanded

# Build the expanded tables (now with up to 15 features)
# all_rules_dt_expanded       = _expand_df_with_lhs_rhs(all_rules_dt, "DT", max_features=15)
all_rules_ripperk_expanded  = _expand_df_with_lhs_rhs(all_rules_ripperk, "RIPPERk", max_features=15)

# Quick peek
# display(all_rules_dt_expanded)
display(all_rules_ripperk_expanded)


,Dataset,Labeling,Feature Encoding,Rule,LHS_features,RHS_label,feature_1_lhs,feature_2_lhs,feature_3_lhs,feature_4_lhs,...,feature_6_lhs,feature_7_lhs,feature_8_lhs,feature_9_lhs,feature_10_lhs,feature_11_lhs,feature_12_lhs,feature_13_lhs,feature_14_lhs,feature_15_lhs
0,BPI15A,declare,baseline,[01_HOOFD_011 = 1.0] --> Label,[01_HOOFD_011 = 1.0],1,01_HOOFD_011 = 1.0,,,,...,,,,,,,,,,
1,BPI15A,declare,bs_data,[01_HOOFD_011 = 1.0] --> Label,[01_HOOFD_011 = 1.0],1,01_HOOFD_011 = 1.0,,,,...,,,,,,,,,,
2,BPI15A,declare,bs_dwd,"[alternate_precedence:(01_HOOFD_011,01_HOOFD_0...","[alternate_precedence:(01_HOOFD_011,01_HOOFD_0...",1,"alternate_precedence:(01_HOOFD_011,01_HOOFD_01...",,,,...,,,,,,,,,,
3,BPI15A,declare,dec_data,"[choice:('01_HOOFD_010', '01_HOOFD_011') = -1....","[choice:('01_HOOFD_010', '01_HOOFD_011') = -1.0]",1,"choice:('01_HOOFD_010', '01_HOOFD_011') = -1.0",,,,...,,,,,,,,,,
4,BPI15A,declare,dec_dwd,"[choice:('01_HOOFD_010', '01_HOOFD_011') = -1....","[choice:('01_HOOFD_010', '01_HOOFD_011') = -1.0]",1,"choice:('01_HOOFD_010', '01_HOOFD_011') = -1.0",,,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438,traffic,sequential,seq_combined_data,"[length_binned_(1.999, 5.0] = 0.0 ∧ paymentAmo...","[length_binned_(1.999, 5.0] = 0.0 ∧ paymentAmo...",1,"length_binned_(1.999, 5.0] = 0.0","paymentAmount|first|continuous_binned_(-0.001,...",,,...,,,,,,,,,,
439,traffic,sequential,seq_combined_data,"[mr[Payment-complete]_binned_(-0.001, 1.0] = 0...","[mr[Payment-complete]_binned_(-0.001, 1.0] = 0...",1,"mr[Payment-complete]_binned_(-0.001, 1.0] = 0.0",mr[Add penalty-complete] = 1.0,,,...,,,,,,,,,,
440,traffic,sequential,seq_combined_data,[mr[Receive Result Appeal from Prefecture-comp...,[mr[Receive Result Appeal from Prefecture-comp...,1,mr[Receive Result Appeal from Prefecture-compl...,"paymentAmount|first|continuous_binned_(-0.001,...",,,...,,,,,,,,,,
441,traffic,sequential,seq_combined_data,"[mra[Payment-complete, Add penalty-complete] =...","[mra[Payment-complete, Add penalty-complete] =...",1,"mra[Payment-complete, Add penalty-complete] = 1.0",,,,...,,,,,,,,,,


## Coverage calculation

In [ ]:
# Recompute rule coverage using normalized labeling folder names and add per-group case IDs
print (os.getcwd())
BASE_DIR = "3.2_binned_features/"

# Map normalized labeling → folder name
LABELING_FOLDER_MAP = {
    "declare": "traffic_decl3_features",
    "sequential": "traffic_mr_tr_features",
    "payload": "traffic_payload_Pay36_features",
}

def _find_ci_subdir(parent: str, target: str) -> str | None:
    """Case-insensitive subdirectory lookup."""
    t = str(target).lower()
    try:
        for d in os.listdir(parent):
            full = os.path.join(parent, d)
            if os.path.isdir(full) and d.lower() == t:
                return full
    except FileNotFoundError:
        return None
    return None

def _resolve_enc_path(dataset: str, labeling: str, encoding: str, base_dir: str) -> str | None:
    """
    Expected layout:
      {base_dir}/{Dataset}/{declare|sequential|payload}_features/{Encoding}
    Uses case-insensitive matching for the labeling/encoding folders.
    """

    if dataset is None or labeling is None or encoding is None:
        return None


    ds_dir = os.path.join(base_dir, str(dataset))

    if not os.path.isdir(ds_dir):
        return None

    lab_norm = str(labeling).strip().lower()
    lab_folder = LABELING_FOLDER_MAP.get(lab_norm) or f"{lab_norm}_features"

    lab_dir = os.path.join(ds_dir, lab_folder)

    if not os.path.isdir(lab_dir):
        lab_dir = _find_ci_subdir(ds_dir, lab_folder)
        if not lab_dir:
            return None

    enc_exact = os.path.join(lab_dir, str(encoding))
    if os.path.isdir(enc_exact):
        return enc_exact
    return _find_ci_subdir(lab_dir, str(encoding))

# --- Helpers for coverage ----------------------------------------------------

def _infer_case_col(df: pd.DataFrame) -> str:
    for c in ["Case_ID", "case:concept:name", "Case ID", "case_id"]:
        if c in df.columns:
            return c
    raise KeyError("No Case ID column found (tried: Case_ID, case:concept:name, Case ID, case_id)")

def _norm_numeric(col: pd.Series) -> pd.Series:
    if col.dtype == bool:
        return col.astype(int)
    out = pd.to_numeric(col, errors="coerce")
    if out.isna().all() and col.dtype == object:
        return col
    return out

NUM_SUFFIX_RE = re.compile(r"_(\-?\d+(?:\.\d+)?)$")

def _match_single_feature(df: pd.DataFrame, feat: str) -> pd.Series:
    """
    Match a single feature token against the event-level feature columns:
      - exact one-hot column
      - base_<num> (equality on numeric/string)
      - binned base_(...) or base_[...]
    Returns a boolean mask.
    """
    feat = str(feat).strip().strip('"').strip("'")

    # A) exact one-hot column
    if feat in df.columns:
        col = _norm_numeric(df[feat])
        return (col == 1) if pd.api.types.is_numeric_dtype(col) else (col.astype(str) == "1")

    # B) base_<num>
    m = NUM_SUFFIX_RE.search(feat)
    if m:
        base_col = feat[:m.start()]
        desired_str = m.group(1)
        desired = float(desired_str)
        if base_col in df.columns:
            col = _norm_numeric(df[base_col])
            if pd.api.types.is_numeric_dtype(col):
                return (col == desired).fillna(False)
            return (col.astype(str) == desired_str).fillna(False)
        # fallback: indicator with suffix
        if feat in df.columns:
            col = _norm_numeric(df[feat])
            return ((col == 1) if pd.api.types.is_numeric_dtype(col) else (col.astype(str) == "1")).fillna(False)

    # C) binned: base_(...) or base_[...]
    pos1 = feat.rfind("_(")
    pos2 = feat.rfind("_[")
    split_pos = max(pos1, pos2)
    if split_pos != -1:
        base_col = feat[:split_pos]
        bin_val  = feat[split_pos + 1 :]  # includes the bracket
        if base_col in df.columns:
            return (df[base_col].astype(str) == bin_val).fillna(False)

    return pd.Series(False, index=df.index)

def _match_rule(df: pd.DataFrame, features: list, rhs_label: int) -> pd.Series:
    """AND all feature matches and enforce RHS label (1=Label, 0=!Label)."""
    mask = pd.Series(True, index=df.index)
    for f in features:
        if f:
            mask &= _match_single_feature(df, f)
            if not mask.any():
                break
    if rhs_label in (0, 1):
        mask &= (pd.to_numeric(df["Label"], errors="coerce") == rhs_label)
    else:
        mask &= False
    return mask

# --- Prepare output frame ----------------------------------------------------

if "all_rules_crm_expanded" not in globals() or not isinstance(all_rules_crm_expanded, pd.DataFrame):
    raise ValueError("Expected all_rules_crm_expanded to be present as a DataFrame.")

ENC_COL = "Encoding" if "Encoding" in all_rules_crm_expanded.columns else \
          ("Feature Encoding" if "Feature Encoding" in all_rules_crm_expanded.columns else None)
if ENC_COL is None:
    raise KeyError("Could not find 'Encoding' or 'Feature Encoding' in all_rules_crm_expanded.")

crm_rules_all_with_coverage = all_rules_crm_expanded.copy()
crm_rules_all_with_coverage["covered_case_ids"] = [[] for _ in range(len(crm_rules_all_with_coverage))]
crm_rules_all_with_coverage["correctly_covered_case_ids"] = [[] for _ in range(len(crm_rules_all_with_coverage))]
crm_rules_all_with_coverage["incorrectly_covered_case_ids"] = [[] for _ in range(len(crm_rules_all_with_coverage))]
crm_rules_all_with_coverage["missing_covered_case_ids"] = [[] for _ in range(len(crm_rules_all_with_coverage))]


crm_rules_all_with_coverage["n_covered_cases"] = 0
crm_rules_all_with_coverage["all_case_ids"] = pd.Series([[]] * len(crm_rules_all_with_coverage), dtype="object")

# --- Compute coverage per (Dataset, Labeling, Encoding) ----------------------

group_cols = [c for c in ["Dataset", "Labeling", ENC_COL] if c in crm_rules_all_with_coverage.columns]

for keys, g in crm_rules_all_with_coverage.groupby(group_cols):
    vals = dict(zip(group_cols, keys))
    ds  = vals.get("Dataset")
    lab = vals.get("Labeling")
    enc = vals.get(ENC_COL)

    enc_path = _resolve_enc_path(ds, lab, enc, BASE_DIR)
    if enc_path is None:
        continue

    csv_files = [f for f in os.listdir(enc_path) if f.lower().endswith(".csv")]
    if not csv_files:
        continue
    csv_path = os.path.join(enc_path, csv_files[0])

    df_enc = pd.read_csv(csv_path)
    if "Label" not in df_enc.columns:
        continue
    case_col = _infer_case_col(df_enc)

    # Cache all case IDs for this (dataset, labeling, encoding)
    all_ids = df_enc[case_col].dropna().astype(str).unique().tolist()
    crm_rules_all_with_coverage.loc[g.index, "all_case_ids"] = pd.Series(
        [all_ids] * len(g), index=g.index, dtype="object"
    )
    pos_ids = (
                    df_enc.loc[pd.to_numeric(df_enc["Label"], errors="coerce") == 1, case_col]
                    .dropna().astype(str).unique().tolist()
                )
    crm_rules_all_with_coverage.loc[g.index, "all_pos_case_ids"] = pd.Series([pos_ids] * len(g), index=g.index, dtype="object")

    # Gather feature columns in numeric order
    feat_cols = [c for c in crm_rules_all_with_coverage.columns if re.fullmatch(r"feature_\d+_lhs", c)]
    feat_cols = sorted(feat_cols, key=lambda x: int(re.findall(r"\d+", x)[0])) if feat_cols else []

    # Evaluate coverage per rule
    for idx, row in g.iterrows():
        feats = [row.get(c, "") for c in feat_cols]
        feats = [f for f in feats if isinstance(f, str) and f.strip() != ""]
        rhs = row.get("RHS_label", None)

        mask = _match_rule(df_enc, feats, rhs)
        covered = df_enc.loc[mask, case_col].dropna().astype(str).unique().tolist()

        crm_rules_all_with_coverage.at[idx, "covered_case_ids"] = covered
        crm_rules_all_with_coverage.at[idx, "n_covered_cases"] = len(covered)
        crm_rules_all_with_coverage.at[idx, "correctly_covered_case_ids"] = [x for x in covered if x in pos_ids]
        crm_rules_all_with_coverage.at[idx, "incorrectly_covered_case_ids"] = [x for x in covered if x not in pos_ids] #false positives
        crm_rules_all_with_coverage.at[idx, "missing_covered_case_ids"] = [x for x in pos_ids if x not in covered] # false negatives



# Optional: order for readability
sort_keys = [c for c in ["Dataset", "Labeling", ENC_COL, "n_covered_cases"] if c in crm_rules_all_with_coverage.columns]
if sort_keys:
    crm_rules_all_with_coverage = crm_rules_all_with_coverage.sort_values(by=sort_keys).reset_index(drop=True)

crm_rules_all_with_coverage=crm_rules_all_with_coverage[crm_rules_all_with_coverage['Dataset']=='traffic']
crm_rules_all_with_coverage.to_csv('crm_rules_all_with_coverage.csv',sep=";")



/Users/heshuis/Library/CloudStorage/OneDrive-TUEindhoven/jupyter-notebooks/causal_deviance_mining-main


In [ ]:
# Compute per-rule coverage for DT & RIPPERK rules and attach all_case_ids
# Supported feature predicates include:
#   01_HOOFD_011 = 0
#   alternate_precedence:(01_HOOFD_011,01_HOOFD_015):Data <= 0.0
#   monitoringResource|first|literal_binned_(560925.0, 12941730.0] = 0/1
#
# Expected folder layout:
#   3.2_binned_features/{Dataset}/{declare_features|sequential_features|payload_features}/{Encoding}/*.csv

BASE_DIR = "3.2_binned_features"
LABELING_FOLDER_MAP = {
    "declare": "traffic_decl3_features",
    "sequential": "traffic_mr_tr_features",
    "payload": "traffic_payload_Pay36_features",
}

# -------- Path & dataframe utilities ----------------------------------------

def _find_ci_subdir(parent: str, target: str) -> str | None:
    """Return a case-insensitive match for subdirectory `target` inside `parent`."""
    t = str(target).lower()
    try:
        for d in os.listdir(parent):
            full = os.path.join(parent, d)
            if os.path.isdir(full) and d.lower() == t:
                return full
    except FileNotFoundError:
        return None
    return None

def _resolve_enc_path(dataset: str, labeling: str, encoding: str, base_dir: str) -> str | None:
    """
    Resolve:
      {base_dir}/{Dataset}/{declare|sequential|payload}_features/{Encoding}
    Labeling/encoding are matched case-insensitively.
    """
    if dataset is None or labeling is None or encoding is None:
        return None

    ds_dir = os.path.join(base_dir, str(dataset))
    if not os.path.isdir(ds_dir):
        return None

    lab_norm = str(labeling).strip().lower()
    lab_folder = LABELING_FOLDER_MAP.get(lab_norm, f"{lab_norm}_features")

    lab_dir = os.path.join(ds_dir, lab_folder)
    if not os.path.isdir(lab_dir):
        lab_dir = _find_ci_subdir(ds_dir, lab_folder)
        if not lab_dir:
            return None

    enc_exact = os.path.join(lab_dir, str(encoding))
    if os.path.isdir(enc_exact):
        return enc_exact
    return _find_ci_subdir(lab_dir, str(encoding))

def _infer_case_col(df: pd.DataFrame) -> str:
    for c in ["Case_ID", "case:concept:name", "Case ID", "case_id"]:
        if c in df.columns:
            return c
    raise KeyError("No Case ID column found (tried: Case_ID, case:concept:name, Case ID, case_id)")

def _to_numeric(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.astype(int)
    return pd.to_numeric(series, errors="coerce")

def _strip_one_layer_quotes(s: str) -> str:
    s = str(s).strip()
    if len(s) >= 2 and s[0] == s[-1] and s[0] in {"'", '"'}:
        return s[1:-1]
    return s

# -------- Feature expression parsing & evaluation ---------------------------

# Pattern: "<col> <op> <val>"  where op ∈ {>=, <=, !=, ==, =, >, <}
_OP_RE = re.compile(r"^(?P<col>.+?)\s*(?P<op>>=|<=|!=|==|=|>|<)\s*(?P<val>.+?)\s*$")

def _parse_feature_expr(expr: str):
    """Return (column, operator, rhs_string) or (None, None, None) if not parsable."""
    s = str(expr).strip()
    m = _OP_RE.match(s)
    if not m:
        return None, None, None
    col = m.group("col").strip()
    op  = "==" if m.group("op") == "=" else m.group("op")
    val_str = _strip_one_layer_quotes(m.group("val"))
    return col, op, val_str

def _coerce_value(val_str: str):
    """Coerce RHS to numeric/bool when possible; otherwise keep as string."""
    low = str(val_str).strip().lower()
    if low in {"true", "false"}:
        return 1 if low == "true" else 0
    try:
        num = float(val_str)
        return int(num) if num.is_integer() else num
    except Exception:
        return val_str

def _cmp_op(series: pd.Series, op: str, rhs):
    """
    Compare series to rhs. Equality works for numeric and string;
    inequalities coerce series to numeric; NaNs evaluate to False.
    """
    if op in ("==", "!="):
        rhs_is_num = isinstance(rhs, (int, float, np.number))
        if rhs_is_num:
            s_num = _to_numeric(series)
            res = (s_num == rhs) if op == "==" else (s_num != rhs)
            if op == "==":
                s_str = series.astype(str).str.strip()
                rhs_str = str(rhs)
                res = res.fillna(s_str == rhs_str)
            else:
                res = res.fillna(True)
            return res.fillna(False)
        else:
            s_str = series.astype(str).str.strip()
            rhs_str = str(rhs).strip()
            return (s_str == rhs_str) if op == "==" else (s_str != rhs_str)

    # Inequalities
    try:
        rhs_num = float(rhs)
    except Exception:
        return pd.Series(False, index=series.index)
    s_num = _to_numeric(series)
    if op == ">":
        return (s_num > rhs_num).fillna(False)
    if op == "<":
        return (s_num < rhs_num).fillna(False)
    if op == ">=":
        return (s_num >= rhs_num).fillna(False)
    if op == "<=":
        return (s_num <= rhs_num).fillna(False)
    return pd.Series(False, index=series.index)

# Split "<base>_(bin)" into (base, "_(bin)")
_BINVAL_SPLIT_RE = re.compile(r"(.+?)(_[(\[][^)\]]+[)\]])$")

def _match_single_feature_dt(df: pd.DataFrame, expr: str) -> pd.Series:
    """
    Evaluate one DT/RIPPERK predicate against df:
      • direct column comparisons (==, !=, >, <, >=, <=)
      • one-hot binned columns: "<col>_(bin) == 0/1"
      • label-encoded bins: base column stores "(bin)" → interpret "== 1" as base == bin
    """
    col, op, val_str = _parse_feature_expr(expr)
    if col is None:
        return pd.Series(False, index=df.index)

    rhs = _coerce_value(val_str)

    # Direct column
    if col in df.columns:
        return _cmp_op(df[col], op, rhs).fillna(False)

    # Binned notation
    m = _BINVAL_SPLIT_RE.match(col)
    if m:
        base_col = m.group(1)
        bin_label = m.group(2)[1:]  # drop leading underscore

        # Try case-insensitive exact column match
        lower_map = {c.lower(): c for c in df.columns}
        candidate = lower_map.get(col.lower())
        if candidate:
            return _cmp_op(df[candidate], op, rhs).fillna(False)

        # Base column holds the bin string
        if base_col in df.columns and op in ("==", "!="):
            base_series = df[base_col].astype(str)
            if isinstance(rhs, (int, float, np.number)) and rhs in (0, 1):
                is_bin = (base_series == bin_label)
                return (is_bin if (op == "==" and rhs == 1) else
                        ~is_bin if (op == "==" and rhs == 0) else
                        ~is_bin if (op == "!=" and rhs == 1) else
                        is_bin).fillna(False)

    return pd.Series(False, index=df.index)

def _match_rule_dt(df: pd.DataFrame, features: list, rhs_label: int) -> pd.Series:
    """Conjoin all feature matches and enforce Label == rhs_label."""
    mask = pd.Series(True, index=df.index)
    for f in features:
        if f:
            mask &= _match_single_feature_dt(df, f)
            if not mask.any():
                break
    if rhs_label in (0, 1):
        mask &= (pd.to_numeric(df["Label"], errors="coerce") == rhs_label)
    else:
        mask &= False
    return mask

def _compute_coverage_for_rules(expanded_df: pd.DataFrame, name_hint: str):
    """
    Compute coverage for a DT/RIPPERK expanded table.
    Adds: covered_case_ids, n_covered_cases, all_case_ids.
    """
    if not isinstance(expanded_df, pd.DataFrame):
        raise ValueError(f"{name_hint}: expanded_df must be a DataFrame.")

    # Encoding column name
    ENC_COL = "Encoding" if "Encoding" in expanded_df.columns else \
              ("Feature Encoding" if "Feature Encoding" in expanded_df.columns else None)
    if ENC_COL is None:
        raise KeyError(f"{name_hint}: Could not find 'Encoding' or 'Feature Encoding'.")

    out = expanded_df.copy()
    out["covered_case_ids"] = [[] for _ in range(len(out))]
    out["n_covered_cases"] = 0
    out["all_case_ids"] = pd.Series([[]] * len(out), dtype="object")
    out["correctly_covered_case_ids"] = [[] for _ in range(len(out))]
    out["incorrectly_covered_case_ids"] = [[] for _ in range(len(out))]
    out["missing_covered_case_ids"] = [[] for _ in range(len(out))]


    group_cols = [c for c in ["Dataset", "Labeling", ENC_COL] if c in out.columns]
    feat_cols = [c for c in out.columns if re.fullmatch(r"feature_\d+_lhs", c)]
    feat_cols = sorted(feat_cols, key=lambda x: int(re.findall(r"\d+", x)[0])) if feat_cols else []

    for keys, g in out.groupby(group_cols):
        vals = dict(zip(group_cols, keys))
        ds  = vals.get("Dataset")
        lab = vals.get("Labeling")
        enc = vals.get(ENC_COL)

        enc_path = _resolve_enc_path(ds, lab, enc, BASE_DIR)
        if enc_path is None:
            continue

        csv_files = [f for f in os.listdir(enc_path) if f.lower().endswith(".csv")]
        if not csv_files:
            continue
        csv_path = os.path.join(enc_path, csv_files[0])

        df_enc = pd.read_csv(csv_path)
        if "Label" not in df_enc.columns:
            continue
        case_col = _infer_case_col(df_enc)

        # Cache case IDs once per (dataset, labeling, encoding)
        all_ids = df_enc[case_col].dropna().astype(str).unique().tolist()
        out.loc[g.index, "all_case_ids"] = pd.Series([all_ids] * len(g), index=g.index, dtype="object")
        pos_ids = (
                    df_enc.loc[pd.to_numeric(df_enc["Label"], errors="coerce") == 1, case_col]
                    .dropna().astype(str).unique().tolist()
                )
        out.loc[g.index, "all_pos_case_ids"] = pd.Series([pos_ids] * len(g), index=g.index, dtype="object")

        for idx, row in g.iterrows():
            feats = [row.get(c, "") for c in feat_cols]
            feats = [f for f in feats if isinstance(f, str) and f.strip() != ""]
            rhs = row.get("RHS_label", None)

            mask = _match_rule_dt(df_enc, feats, rhs)
            covered = df_enc.loc[mask, case_col].dropna().astype(str).unique().tolist()

            out.at[idx, "covered_case_ids"] = covered
            out.at[idx, "n_covered_cases"] = len(covered)
            out.at[idx, "correctly_covered_case_ids"] = [x for x in covered if x in pos_ids]
            out.at[idx, "incorrectly_covered_case_ids"] = [x for x in covered if x not in pos_ids] #false positives
            out.at[idx, "missing_covered_case_ids"] = [x for x in pos_ids if x not in covered] # false negatives


    sort_keys = [c for c in ["Dataset", "Labeling", ENC_COL, "n_covered_cases"] if c in out.columns]
    if sort_keys:
        out = out.sort_values(by=sort_keys).reset_index(drop=True)
    return out

# ---- Build coverage tables --------------------------------------------------
# if 'all_rules_dt_expanded' in globals():
#     all_rules_dt_with_coverage = _compute_coverage_for_rules(all_rules_dt_expanded, "DT")
# else:
#     raise ValueError("all_rules_dt_expanded not found. Run the expansion cell first.")

if 'all_rules_ripperk_expanded' in globals():
    all_rules_ripperk_with_coverage = _compute_coverage_for_rules(all_rules_ripperk_expanded, "RIPPERK")
else:
    raise ValueError("all_rules_ripperk_expanded not found. Run the expansion cell first.")

print (all_rules_ripperk_with_coverage)
all_rules_ripperk_with_coverage=all_rules_ripperk_with_coverage[all_rules_ripperk_with_coverage['Dataset']=='traffic']
all_rules_ripperk_with_coverage.to_csv("all_rules_ripperk_with_coverage.csv",sep=";")

     Dataset    Labeling   Feature Encoding  \
0     BPI15A     declare           baseline   
1     BPI15A     declare            bs_data   
2     BPI15A     declare             bs_dwd   
3     BPI15A     declare           dec_data   
4     BPI15A     declare            dec_dwd   
..       ...         ...                ...   
438  traffic  sequential  seq_combined_data   
439  traffic  sequential  seq_combined_data   
440  traffic  sequential  seq_combined_data   
441  traffic  sequential  seq_combined_data   
442  traffic  sequential  seq_combined_data   

                                                  Rule  \
0                       [01_HOOFD_011 = 1.0] --> Label   
1                       [01_HOOFD_011 = 1.0] --> Label   
2    [alternate_precedence:(01_HOOFD_011,01_HOOFD_0...   
3    [choice:('01_HOOFD_010', '01_HOOFD_011') = -1....   
4    [choice:('01_HOOFD_010', '01_HOOFD_011') = -1....   
..                                                 ...   
438  [mr[Receive Result Appea

In [ ]:
import ast 


def f1_score_row(row):
    TP = len(row['correctly_covered_case_ids'])
    FP = len(row['incorrectly_covered_case_ids'])
    FN = len(row['missing_covered_case_ids'])
    
    # handle edge cases
    if TP == 0 and FP == 0 and FN == 0:
        return 0.0
    
    # precision and recall
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    
    # compute F1
    if precision + recall == 0:
        return 0.0
    
    return 2 * precision * recall / (precision + recall)

def precision_score_row(row):
    TP = len(row['correctly_covered_case_ids'])
    FP = len(row['incorrectly_covered_case_ids'])
    # FN = len(row['missing_covered_case_ids'])
    
    # handle edge cases
    if TP == 0 and FP == 0:
        return 0.0
    
    # precision and recall
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    # recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

    return precision

def recall_score_row(row):
    TP = len(row['correctly_covered_case_ids'])
    # FP = len(row['incorrectly_covered_case_ids'])
    FN = len(row['missing_covered_case_ids'])
    
    # handle edge cases
    if TP == 0 and FN == 0:
        return 0.0
    
    # precision and recall
    # precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

    return recall


def concat_lists(lists):
    new_list = []
    for l in lists:
        new_list.extend(set(l)-set(new_list) )
    new_list.sort()
    return new_list

df_ripperk=all_rules_ripperk_with_coverage
df_ripperk=df_ripperk[df_ripperk.get("RHS_label", 1) == 1]
df_ripperk['LHS_features_list']=df_ripperk["LHS_features"].apply(split_lhs_items_dt)
df_ripperk['len_ripperk_rule']=df_ripperk['LHS_features_list'].apply(len)
# df_agg_ripperk = df_ripperk.groupby(['Dataset', 'Labeling', 'Feature Encoding'], as_index=False).agg({   'Rule':lambda x:list(x),   'covered_case_ids': lambda lists: concat_lists(lists), 'all_case_ids': lambda lists: concat_lists(lists)})
df_agg_ripperk = df_ripperk.groupby(['Dataset', 'Labeling', 'Feature Encoding'], as_index=False).agg({   'Rule':lambda x:list(x),   'covered_case_ids': lambda x: list(x), 'all_case_ids': lambda lists: concat_lists(lists), 'all_pos_case_ids': lambda lists: concat_lists(lists), 'correctly_covered_case_ids':lambda lists: concat_lists(lists), 'incorrectly_covered_case_ids':lambda lists: concat_lists(lists),'missing_covered_case_ids':lambda lists: concat_lists(lists), 'LHS_features_list': lambda lists:concat_lists(lists), 'len_ripperk_rule': lambda x: x.mean()})

df_agg_ripperk['ripperk_missing_covered_case_ids'] = df_agg_ripperk.apply(
    lambda row: [x for x in row['all_pos_case_ids'] 
                 if x not in row['correctly_covered_case_ids']],
    axis=1
)
df_agg_ripperk['precision'] = df_agg_ripperk.apply(precision_score_row, axis=1)
df_agg_ripperk['recall'] = df_agg_ripperk.apply(recall_score_row, axis=1)
df_agg_ripperk['f1_score'] = df_agg_ripperk.apply(f1_score_row, axis=1)

df_agg_ripperk['unique_covered_ripperk_clusters_count'] = df_agg_ripperk['covered_case_ids'].apply(lambda x: len(set(tuple(sublist) for sublist in x)))
df_agg_ripperk['ripperk_features']=df_agg_ripperk['LHS_features_list'].apply(len)

df_agg_ripperk.rename(columns={'Rule':'ripperk_rules','covered_case_ids':'ripperk_covered_case_ids','all_case_ids':'ripperk_all_case_ids','all_pos_case_ids':'ripperk_all_pos_case_ids','correctly_covered_case_ids':'ripperk_correctly_covered_case_ids','incorrectly_covered_case_ids':'ripperk_incorrectly_covered_case_ids','missing_covered_case_ids':'ripperk_missing_covered_case_ids','LHS_features_list':'ripperk_LHS_features','precision' : 'ripperk_precision','recall' : 'ripperk_recall', 'f1_score' : 'ripperk_f1_score'},inplace=True)
df_agg_ripperk['nr ripperk rules']=df_agg_ripperk['ripperk_rules'].apply(len)
df_agg_ripperk.to_csv('df_agg_ripperk-pos.csv',sep=";")

df_crm=crm_rules_all_with_coverage
df_crm=df_crm[df_crm.get("RHS_label", 1) == 1]
# test_coverage_crm=df_crm.groupby(['Dataset','Labeling','Feature Encoding'])['covered_case_ids'].agg(list).reset_index()
# # test_coverage_crm['different_n_covered_cases']=test_coverage_crm['covered_case_ids'].map(set)
# test_coverage_crm['unique_count'] = test_coverage_crm['covered_case_ids'].apply(lambda x: len(set(tuple(sublist) for sublist in x)))
# test_coverage_crm['clusters_covered_cases']=test_coverage_crm['covered_case_ids'].map(len)
# test_coverage_crm.to_csv('test_crm_rules_all_with_coverage.csv',sep=";")

# df_agg_crm=df_crm.groupby(['Dataset', 'Labeling', 'Feature Encoding'], as_index=False).agg({  'Rule':lambda x:list(x),  'covered_case_ids': lambda lists: concat_lists(lists), 'all_case_ids': lambda lists: concat_lists(lists)})
df_crm['LHS_features'] = df_crm['LHS_features'].apply(ast.literal_eval)
df_crm['len_CRM_rule'] = df_crm['LHS_features'].apply(len)

df_agg_crm=df_crm.groupby(['Dataset', 'Labeling', 'Feature Encoding'], as_index=False).agg({  'Rule':lambda x:list(x),  'covered_case_ids': lambda x:list(x), 'all_case_ids': lambda lists: concat_lists(lists), 'all_pos_case_ids': lambda lists: concat_lists(lists), 'correctly_covered_case_ids':lambda lists: concat_lists(lists), 'incorrectly_covered_case_ids':lambda lists: concat_lists(lists), 'LHS_features': lambda x: [item for sublist in x for item in sublist ],'len_CRM_rule': lambda x: x.mean()})
df_agg_crm['missing_covered_case_ids'] = df_agg_crm.apply(
    lambda row: [x for x in row['all_pos_case_ids'] 
                 if x not in row['correctly_covered_case_ids']],
    axis=1
)
df_agg_crm['precision'] = df_agg_crm.apply(precision_score_row, axis=1)
df_agg_crm['recall'] = df_agg_crm.apply(recall_score_row, axis=1)
df_agg_crm['f1_score'] = df_agg_crm.apply(f1_score_row, axis=1)

df_agg_crm['len_covered_case_ids']=df_agg_crm['covered_case_ids'].apply(lambda x:[len(s) for s in x])
df_agg_crm['unique_covered_CRM_clusters_count'] = df_agg_crm['covered_case_ids'].apply(lambda x: len(set(tuple(sublist) for sublist in x)))
df_agg_crm['crm_features']=df_agg_crm['LHS_features'].apply(len)


df_agg_crm.rename(columns={'Rule':'CRM_rules','covered_case_ids':'crm_covered_case_ids','all_case_ids':'crm_all_case_ids','all_pos_case_ids':'crm_all_pos_case_ids','correctly_covered_case_ids':'crm_correctly_covered_case_ids','incorrectly_covered_case_ids':'crm_incorrectly_covered_case_ids','missing_covered_case_ids':'crm_missing_covered_case_ids','LHS_features':'CRM_LHS_features','precision' : 'crm_precision','recall' : 'crm_recall','f1_score' : 'crm_f1_score'},inplace=True)
df_agg_crm['nr CRM rules']=df_agg_crm['CRM_rules'].apply(len)
df_agg_crm.to_csv('df_agg_crm-pos.csv',sep=";")

# df_agg_crm['CRM_LHS_features'].head(5)
# df_agg_ripperk.head(5)


/var/folders/n1/8vyc_4kj42z487kp50cds42c0000gn/T/ipykernel_14085/3380607492.py:93: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_crm['LHS_features'] = df_crm['LHS_features'].apply(ast.literal_eval)
/var/folders/n1/8vyc_4kj42z487kp50cds42c0000gn/T/ipykernel_14085/3380607492.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_crm['len_CRM_rule'] = df_crm['LHS_features'].apply(len)


In [ ]:
from itertools import chain

df_merge=df_agg_ripperk.merge(df_agg_crm,on=['Dataset','Labeling','Feature Encoding'])
df_merge['equal_all_case_ids'] = df_merge.apply(lambda row: row['crm_all_case_ids'] == row['ripperk_all_case_ids'], axis=1)
df_merge['all_crm_covered_case_ids']= df_merge['crm_covered_case_ids'].apply(lambda x: list(set(chain.from_iterable(x))))
df_merge['all_ripperk_covered_case_ids']= df_merge['ripperk_covered_case_ids'].apply(lambda x: list(set(chain.from_iterable(x))))

df_merge['crm_covered_subset_ripperk_covered'] = df_merge.apply(lambda row: set(row['all_crm_covered_case_ids']).issubset(row['all_ripperk_covered_case_ids']), axis=1)
df_merge['ripperk_covered_subset_crm_covered'] = df_merge.apply(lambda row: set(row['all_ripperk_covered_case_ids']).issubset(row['all_crm_covered_case_ids']), axis=1)

df_merge['all_ripperk_covered_case_ids_len']=df_merge['all_ripperk_covered_case_ids'].apply(len)
df_merge['ripperk_all_pos_case_ids_len']=df_merge['ripperk_all_pos_case_ids'].apply(len)
# df_merge['ripperk_recall']=df_merge['all_ripperk_covered_case_ids_len']/df_merge['ripperk_all_pos_case_ids_len']


df_merge['all_crm_covered_case_ids_len']=df_merge['all_crm_covered_case_ids'].apply(len)
df_merge['crm_all_pos_case_ids_len']=df_merge['crm_all_pos_case_ids'].apply(len)
# df_merge['crm_recall']=df_merge['all_crm_covered_case_ids_len']/df_merge['crm_all_pos_case_ids_len']
# df_merge['CRM rules']=df_merge['CRM_rules'].apply(len)
# df_merge['RIPPERk rules']=df_merge['ripperk_rules'].apply(len)

df_merge.to_csv('merged-coverage-pos.csv',sep=';')

df_feature=df_merge
df_feature=df_feature[['Dataset','Labeling','Feature Encoding','nr ripperk rules','ripperk_features','nr CRM rules','crm_features','len_ripperk_rule','len_CRM_rule','ripperk_LHS_features','CRM_LHS_features']]
df_feature.to_csv('features.csv',sep=';')

# df_latex=df_merge
# df_latex.drop(columns=['Dataset','len_covered_case_ids','ripperk_rules','ripperk_covered_case_ids','ripperk_all_case_ids','ripperk_all_pos_case_ids','CRM_rules','crm_covered_case_ids','crm_all_case_ids','crm_all_pos_case_ids','equal_all_case_ids','all_crm_covered_case_ids','all_ripperk_covered_case_ids','crm_covered_subset_ripperk_covered','all_ripperk_covered_case_ids_len','ripperk_all_pos_case_ids_len','all_crm_covered_case_ids_len','crm_all_pos_case_ids_len'],inplace=True)

# cov_cols=['ripperk_recall','crm_recall']
# def fmt2(x):
#     return "" if pd.isna(x) else f"{float(x):.2f}"
# formatters = {c: fmt2 for c in cov_cols if c in df_latex.columns}

# latex_path="./exp.tex"
# with open(latex_path, "w", encoding="utf-8") as f:
#     f.write(df_latex.to_latex(index=False, escape=False, formatters=formatters))